## Пример запуска персонального AI-ассистента с BFS-алгоритмом поиска

In [1]:
import sys
# TO CHANGE
BASEDIR = "../../../"
sys.path.insert(0, BASEDIR)

In [2]:
from src import PersonalAI, PersonalAIConfig, QAPipelineConfig, MemPipelineConfig, \
      GraphModelConfig, EmbeddingsModelConfig, EmbedderModelConfig

from src.db_drivers import GraphDriverConfig, VectorDriverConfig
from src.db_drivers.graph_driver import DEFAULT_INMEMORYGRAPH_CONFIG
from src.db_drivers.vector_driver import VectorDBConnectionConfig

from src.qa_pipeline.knowledge_retriever import BFSSearchConfig
from src.qa_pipeline import QueryLLMParserConfig, KnowledgeComparatorConfig, KnowledgeRetrieverConfig, QALLMGeneratorConfig

from src.memorize_pipeline import LLMExtractorConfig, LLMUpdatorConfig

from src.utils import Logger

/home/dzigen/Desktop/PersonalAI/pai_venv/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


### 1. Задаём конфигурацию графа знаний

In [ ]:
LANGUAGE = 'auto'

In [4]:
RETRIEVER_NAME = 'bfs'
RETRIEVER_HYPERP = BFSSearchConfig(
    strict_filter = False, hyper_episodic_num = 15,
    chain_triplets_num = 25, other_triplets_num = 6)

In [ ]:
# Graph model configuration
GRAPH_STORAGE_CONFIG = GraphDriverConfig(db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)
GRAPH_MODEL_CONFIG = GraphModelConfig(driver_config=GRAPH_STORAGE_CONFIG)

In [ ]:
# Vector model configuration
NODES_DB_PATH = '../../data/graph_structures/vectorized_nodes/testing'
TRIPLETS_DB_PATH = '../../data/graph_structures/vectorized_triplets/testing'
NEED_TO_CLEAR = True

VECTOR_NODES_STORAGE_CONFIG = VectorDriverConfig(db_config=VectorDBConnectionConfig(path=NODES_DB_PATH, need_to_clear=NEED_TO_CLEAR))
VECTOR_TRIPLETS_STIRAGE_CONFIG = VectorDriverConfig(db_config=VectorDBConnectionConfig(path=TRIPLETS_DB_PATH, need_to_clear=NEED_TO_CLEAR))

DEVICE = 'cuda'
EMBEDDER_MODEL_PATH = '../../models/intfloat/multilingual-e5-small'
EMBEDDER_MODEL_CONFIG = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH, device=DEVICE)

VECTOR_MODEL_CONFIG = EmbeddingsModelConfig(
    nodesdb_driver_config=VECTOR_NODES_STORAGE_CONFIG,
    tripletsdb_driver_config=VECTOR_TRIPLETS_STIRAGE_CONFIG,
    embedder_config=EMBEDDER_MODEL_CONFIG)

In [ ]:
# QA-pipeline configuration
QA_PIPELINE_CONFIG = QAPipelineConfig(
    query_parser_config=QueryLLMParserConfig(lang=LANGUAGE),
    knowledge_comparator_config=KnowledgeComparatorConfig(),
    knowledge_retriever_config=KnowledgeRetrieverConfig(
        retriever_method=RETRIEVER_NAME,retriever_config=RETRIEVER_HYPERP),
    answer_generator_config=QALLMGeneratorConfig(lang=LANGUAGE))

In [ ]:
# Memorize-pipeline configuration
MEM_PIPELINE_CONFIG = MemPipelineConfig(
    xtractor_config=LLMExtractorConfig(lang=LANGUAGE),
    updator_config=LLMUpdatorConfig(lang=LANGUAGE))

In [5]:
PERSONALAI_CONFIG = PersonalAIConfig(
    graph_struct_config=GRAPH_MODEL_CONFIG,
    embedds_struct_config=VECTOR_MODEL_CONFIG,
    qa_pipeline_config=QA_PIPELINE_CONFIG,
    mem_pipeline_config=MEM_PIPELINE_CONFIG,
    log=Logger('log/main'))

#### 2. Инициализируем персонального ассистента

In [6]:
personalai = PersonalAI(config=PERSONALAI_CONFIG)

No sentence-transformers model found with name ../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


#### 3. Пример работы Memorize-конвейера

In [8]:
messages = [
    "Mikhail Menshchikov is currently a second-year master's student at ITMO.",
    "Mikhail Menshchikov is studying in the Master's program 'Deep Learning and Generative AI'",
    "Mikhail Menshchikov completed his bachelor's degree at Petrozavodsk State University",
    "Petrozavodsk State University is where Mikhail Menshchikov received his bachelor's degree.",
    "Mikhail Menshchikov studied at Petrozavodsk State University and received a bachelor's degree."]
properties = [dict() for _ in range(len(messages))]

In [ ]:
for text, prop in zip(messages, properties):
    _, info = personalai.update_memory(text, prop)

#### 4. Пример работы QA-конвейера

In [10]:
answer, info = personalai.answer_question("What program is Mikhail Menshchikov studying for his master's degree?")
print(answer)

Deep Learning and Generative AI


In [11]:
answer, info = personalai.answer_question("Where did Mikhail Menshchikov receive his bachelor's degree?")
print(answer)

Petrozavodsk State University
